In [1]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import f1_score, recall_score, confusion_matrix, roc_auc_score, make_scorer, roc_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, ParameterGrid
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import xgboost as xgb
from sklearn.utils import resample

In [2]:
# Load the Excel file
file_path =  r"D:\Singapore\Alzheimer\R files\final data_new\Variable list-11MAY2025.xlsx"
df_Variables = pd.read_excel(file_path)

In [3]:
file_path_1 = r"D:\Singapore\Alzheimer\R files\final data_new\imoutations_diff_ks\final_data_imputed_k1.csv"
df_Data = pd.read_csv(file_path_1)

In [4]:
# Group features based on the "Group" column
demographics = df_Variables[df_Variables['Group'] == 'Demographic']['Predictor'].tolist()
clinical_cognitive = df_Variables[df_Variables['Group'] == 'Clinical/Cognitive']['Predictor'].tolist()
biomarker = df_Variables[df_Variables['Group'] == 'Biomarker']['Predictor'].tolist()
neuroimaging = df_Variables[df_Variables['Group'] == 'Neuroimaging']['Predictor'].tolist()

# Optional: Print or check lists
print("Demographics:", demographics)
print("Clinical/Cognitive:", clinical_cognitive)
print("Biomarker:", biomarker)
print("Neuroimaging:", neuroimaging)

Demographics: ['AGE', 'PTGENDER', 'PTEDUCAT', 'PTRACCAT', 'PTMARRY']
Clinical/Cognitive: ['CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'RAVLT.immediate', 'RAVLT.learning', 'RAVLT.forgetting', 'RAVLT.perc.forgetting', 'LDELTOTAL', 'TRABSCOR', 'FAQ', 'EcogPtTotal', 'MOCA_bl', 'MMSE_bl']
Biomarker: ['FDG', 'AV45', 'ABETA', 'TAU', 'PTAU']
Neuroimaging: ['Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal', 'Fusiform', 'MidTemp', 'ICV']


In [5]:
# Group combinations
group_1 = demographics + clinical_cognitive
group_2 = demographics + biomarker
group_3 = demographics + clinical_cognitive + biomarker
group_4 = neuroimaging

In [6]:
group_3_tvae1 = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\TVAE\group_1_3\synthetic_group_3_TVAE_1x.csv"
group_3_tvae1_Data = pd.read_csv(group_3_tvae1)

group_3_tvae2 = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\TVAE\group_1_3\synthetic_group_3_TVAE_2x.csv"
group_3_tvae2_Data = pd.read_csv(group_3_tvae2)

group_3_tvae3 = r"D:\Singapore\Alzheimer\R files\final data_new\synthetic_data\TVAE\group_1_3\synthetic_group_3_TVAE_3x.csv"
group_3_tvae3_Data = pd.read_csv(group_3_tvae3)

In [7]:
def TRTS_group1_group3(group_columns, df_real, df_syn, group_name,
                      target_col="convert_Within_3Years",
                      method="RF",
                      random_state=100):
    """
    Train on synthetic, test on real with Youden threshold calibration,
    RF or XGB, CV metrics, confidence intervals.
    """
    
    X_train = df_real[group_columns]
    y_train = df_real[target_col]

    X_test = df_syn[group_columns]
    y_test = df_syn[target_col]

    # -------------------------
    # CV
    # -------------------------
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    # -------------------------
    # Model setup
    # -------------------------
    if method.upper() == "RF":
        model = RandomForestClassifier(random_state=random_state, class_weight=  'balanced')
        param_dist = {
            'n_estimators': [200, 300],
            'max_depth': [2, 3],
            'min_samples_split': [15, 20, 30],
            'min_samples_leaf': [8, 10, 15],
            'max_features': ['log2'],
            'bootstrap': [True],
            'max_samples': [0.5, 0.7]
        }
    elif method.upper() == "XGBOOST":
        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()
        scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1

        model = xgb.XGBClassifier(
            eval_metric='logloss',
            random_state=random_state,
            scale_pos_weight=scale_pos_weight
        )
        param_dist = {
      
            'n_estimators': [50, 100, 150],
            'max_depth': [1, 2],
            'learning_rate': [0.05, 0.1],
            'min_child_weight': [2, 3, 4],
            'subsample': [0.6, 0.8],
            'colsample_bytree': [0.6, 0.8],
            'gamma': [0, 0.1, 0.3],
            'reg_alpha': [0, 0.1, 0.3],
            'reg_lambda': [0, 0.01, 0.3]
        }
    else:
        raise ValueError("Method must be RF or XGBoost")

    # -------------------------
    # Metrics
    # -------------------------
    def specificity_score(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0

    scoring = {
        'roc_auc': 'roc_auc',
        'f1': make_scorer(f1_score),
        'recall': make_scorer(recall_score),
        'specificity': make_scorer(specificity_score)
    }

    # -------------------------
    # Hyperparameter tuning
    # -------------------------
    search = RandomizedSearchCV(
        model,
        param_distributions=param_dist,
        cv=cv,
        scoring=scoring,
        refit='roc_auc',
        random_state=random_state
    )

    search.fit(X_train, y_train)
    best_params = search.best_params_

    uncalibrated_model = xgb.XGBClassifier(**best_params, random_state=100,  scale_pos_weight=scale_pos_weight) if method.upper() == "XGBOOST" else RandomForestClassifier(**best_params, random_state=100, class_weight='balanced')
    calibrated_model = CalibratedClassifierCV(uncalibrated_model, method='isotonic', cv=3)
    calibrated_model.fit(X_train, y_train)

    y_train_proba = calibrated_model.predict_proba(X_train)[:, 1]
    y_test_proba = calibrated_model.predict_proba(X_test)[:, 1]

    # -------------------------
    # Youden threshold (train CV)
    # -------------------------
    youden_thresholds = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        if method.upper() == "XGBOOST":
            model_cv = xgb.XGBClassifier(**best_params, random_state=random_state,
                                         scale_pos_weight=scale_pos_weight)
        else:
            model_cv = RandomForestClassifier(**best_params, random_state=random_state,
                                              class_weight='balanced')
        model_cv_cal = CalibratedClassifierCV(model_cv, method='isotonic', cv=5)
        model_cv_cal.fit(X_tr, y_tr)
        y_val_proba = model_cv_cal.predict_proba(X_val)[:, 1]
        fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
        
        # ---------- CHANGED PART ----------
        youden = tpr - fpr
        valid_idx = np.where(np.isfinite(thresholds))[0]
        best_thresh = thresholds[valid_idx[np.argmax(youden[valid_idx])]]
        # ---------------------------------

        youden_thresholds.append(best_thresh)

    optimal_threshold = np.mean(youden_thresholds)

    # -------------------------
    # Predictions
    # -------------------------
    y_train_pred = (y_train_proba >= optimal_threshold).astype(int)
    y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

    # -------------------------
    # Metrics
    # -------------------------
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(y_train, y_train_pred).ravel()
    tn_te, fp_te, fn_te, tp_te = confusion_matrix(y_test, y_test_pred).ravel()

    results = {
        'group': group_name,
        'method': method,
        'synthetic_size': len(df_syn),
        'best_params': best_params,
        'cv_auc': round(search.cv_results_['mean_test_roc_auc'][search.best_index_], 3),
        'cv_f1': round(search.cv_results_['mean_test_f1'][search.best_index_], 3),
        'cv_recall': round(search.cv_results_['mean_test_recall'][search.best_index_], 3),
        'cv_specificity': round(search.cv_results_['mean_test_specificity'][search.best_index_], 3),
        'train_auc': round(roc_auc_score(y_train, y_train_proba), 3),
        'train_f1': round(f1_score(y_train, y_train_pred), 3),
        'train_recall': round(recall_score(y_train, y_train_pred), 3),
        'train_specificity': round(tn_tr / (tn_tr + fp_tr), 3), 
        'test_auc': round(roc_auc_score(y_test, y_test_proba), 3),
        'test_f1': round(f1_score(y_test, y_test_pred), 3),
        'test_recall': round(recall_score(y_test, y_test_pred), 3),
        'test_specificity': round(tn_te / (tn_te + fp_te), 3),
        'youden_threshold': optimal_threshold
    }
    return results

In [8]:
results_xgb_group_3_tvae1 = TRTS_group1_group3(
    group_columns = group_3,
    df_real = df_Data,
    df_syn = group_3_tvae1_Data,
    group_name = "group_3",
    method = "XGBoost"
)

results_xgb_group_3_tvae2 = TRTS_group1_group3(
    group_columns = group_3,
    df_real = df_Data,
    df_syn = group_3_tvae2_Data,
    group_name = "group_3",
    method = "XGBoost"
)

results_xgb_group_3_tvae3 = TRTS_group1_group3(
    group_columns = group_3,
    df_real = df_Data,
    df_syn = group_3_tvae3_Data,
    group_name = "group_3",
    method = "XGBoost"
)

In [9]:
all_results_group_3_xgb = [results_xgb_group_3_tvae1, results_xgb_group_3_tvae2, results_xgb_group_3_tvae3]
all_results_group_3_xgb

[{'group': 'group_3',
  'method': 'XGBoost',
  'synthetic_size': 473,
  'best_params': {'subsample': 0.6,
   'reg_lambda': 0,
   'reg_alpha': 0,
   'n_estimators': 150,
   'min_child_weight': 4,
   'max_depth': 2,
   'learning_rate': 0.05,
   'gamma': 0.3,
   'colsample_bytree': 0.6},
  'cv_auc': 0.937,
  'cv_f1': 0.676,
  'cv_recall': 0.841,
  'cv_specificity': 0.879,
  'train_auc': 0.981,
  'train_f1': 0.713,
  'train_recall': 1.0,
  'train_specificity': 0.849,
  'test_auc': 0.969,
  'test_f1': 0.321,
  'test_recall': 1.0,
  'test_specificity': 0.842,
  'youden_threshold': 0.17724359422922134},
 {'group': 'group_3',
  'method': 'XGBoost',
  'synthetic_size': 946,
  'best_params': {'subsample': 0.6,
   'reg_lambda': 0,
   'reg_alpha': 0,
   'n_estimators': 150,
   'min_child_weight': 4,
   'max_depth': 2,
   'learning_rate': 0.05,
   'gamma': 0.3,
   'colsample_bytree': 0.6},
  'cv_auc': 0.937,
  'cv_f1': 0.676,
  'cv_recall': 0.841,
  'cv_specificity': 0.879,
  'train_auc': 0.981,
  